# 00 - environment check

**Run this first, and again whenever the container image changes.** It is cheap and it
catches the one thing that would waste a day.

`tensorflow_pbs:23.07-tf2-py3` is NGC 23.07, which ships **TensorFlow 2.13 / Keras 2.x**.
Gate 1d was verified locally on Keras **3.15**, and the two APIs differ in exactly the
places this model uses them - `keras.ops`, masked attention keywords, pooling signatures.
`src/mapguard/models/stage2.py` is written to build on both. This notebook proves it on
the image that will actually train.

It needs no data and no credentials.

In [ ]:
# Unzip the bundle uploaded through the Altair portal, then work from it.
# Everything real lives in src/mapguard/ - see CONTRIBUTING.md.
import glob, json, os, sys, zipfile, pathlib

BUNDLE = next(iter(sorted(glob.glob("*.zip")) + sorted(glob.glob("/data/**/*.zip", recursive=True))), None)
if BUNDLE and not pathlib.Path("src/mapguard").exists():
    with zipfile.ZipFile(BUNDLE) as z:
        z.extractall(".")
    print("unpacked", BUNDLE)

sys.path.insert(0, "src")
TASK = [p for p in glob.glob("*/BUNDLE_MANIFEST.json")]
if TASK:
    manifest = json.load(open(TASK[0]))
    print(json.dumps({k: manifest[k] for k in ("task","git_sha","git_dirty","generated")}, indent=2))
    if manifest.get("git_dirty"):
        print("WARNING: bundle was built from a dirty tree; git_sha does not fully describe it")

## GPU and versions

Ask for **1 GPU**. It is a 40 GB MIG slice of an H100 and the others belong to other people.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
import sys, platform
import numpy as np
import tensorflow as tf
from tensorflow import keras

print("python  ", platform.python_version())
print("tf      ", tf.__version__)
print("keras   ", keras.__version__)
print("numpy   ", np.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs    ", gpus)
assert gpus, "no GPU visible - resubmit the job with Number of GPUs per Node = 1"

## Does the model build here?

Every branch, then a few subsets, since the ablation grid is the same function with a
different `branches` tuple.

In [ ]:
from mapguard.models.stage2 import build_model, focal_loss, verify_masking

CFG = dict(max_edits=32, edit_features=14, text_len=128, vocab_size=56,
           user_steps=32, user_features=4, tabular_features=41)

for branches in [("B5",), ("B2","B5"), ("B3","B5"), ("B4","B5"), ("B1","B2","B3","B4","B5")]:
    m = build_model(branches=branches, raster=16 if "B1" in branches else None, **CFG)
    print(f"{str(branches):34} inputs={len(m.inputs)}  params={m.count_params():>9,}")

## Does it train, and does masking hold?

The masking check is the one that matters. A model that silently attends to padding trains
fine, scores fine, and is wrong - and the failure presents three weeks later as
"geometry did not help".

We perturb **only the padded slots** by 50x noise. A correct model does not move at all.

In [ ]:
import numpy as np
model = build_model(branches=("B1","B2","B3","B4","B5"), raster=16, **CFG)
model.compile(optimizer="adam",
              loss={"p_damaging": focal_loss()},
              metrics={"p_damaging": ["accuracy"]})

rng = np.random.default_rng(20260908)
n, K = 64, CFG["max_edits"]
real = rng.integers(1, K + 1, n)
mask = (np.arange(K)[None, :] < real[:, None]).astype("float32")

x = dict(
    edit_features=(rng.normal(size=(n, K, CFG["edit_features"])) * mask[:, :, None]).astype("float32"),
    edit_mask=mask,
    geometry=(rng.normal(size=(n, K, 16, 16, 2)) * mask[:, :, None, None, None]).astype("float32"),
    text=rng.integers(0, CFG["vocab_size"], (n, CFG["text_len"])).astype("int32"),
    user_sequence=rng.normal(size=(n, CFG["user_steps"], 4)).astype("float32"),
    user_mask=np.ones((n, CFG["user_steps"]), "float32"),
    tabular=rng.normal(size=(n, CFG["tabular_features"])).astype("float32"),
)
order = [t.name.split(":")[0] for t in model.inputs]
y = rng.integers(0, 2, n).astype("float32")

hist = model.fit([x[k] for k in order],
                 {"p_damaging": y, "edit_attention_weight": np.zeros((n, K), "float32")},
                 epochs=1, batch_size=16, verbose=0)
print("trained one epoch, loss =", hist.history["loss"][0])

check = verify_masking(model, x)
print("masking:", check)
assert check["masking_holds"], "MASKING LEAKS - do not train on this image until fixed"

## Three outputs

`p(damaging)`, per-edit attention weight, and (at training time) branch attribution.
Output 2 is what tells a reviewer *which edit* to open.

In [ ]:
p, w = model.predict([x[k] for k in order], verbose=0)
print("p(damaging)      ", p.shape)
print("edit weights     ", w.shape)
row = 0
print(f"row {row}: {int(real[row])} real edits, most suspicious index = {int(w[row][:real[row]].argmax())}")

## Save the evidence

Download `results/cluster_env_check.json` and commit it. A gate with no file did not run.

In [ ]:
import json, os, subprocess, datetime
os.makedirs("results", exist_ok=True)
out = {
    "generated": datetime.datetime.now().isoformat(timespec="seconds"),
    "where": "SVKM AI/ML cluster, Altair Access",
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "gpus": [d.name for d in gpus],
    "model_built": True,
    "trained_one_epoch": True,
    "masking": check,
    "verdict": "PASS" if check["masking_holds"] else "FAIL",
}
json.dump(out, open("results/cluster_env_check.json", "w"), indent=2)
print(json.dumps(out, indent=2))